# NepalDTM-G: Bagmati 1 m generation on Colab

Runs the second half of the Bagmati Province tiles (last-first) while a second machine works first-first.
Each finished tile is uploaded to the R2 bucket `nepaldtm-g-1m` at `bagmati/tiles/`; tiles already in R2 are skipped, so the notebook can be
re-run after a disconnect without losing more than the tile in progress.

**Before you start**
1. Runtime → Change runtime type → **GPU** (A100 if available, else L4).
2. Add three Colab secrets (key icon, left bar), with notebook access on (the API token needs Object Read & Write on bucket `nepaldtm-g-1m`):
   `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_ENDPOINT` (`https://<account-id>.r2.cloudflarestorage.com`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import userdata
for k in ("R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_ENDPOINT"):
    os.environ[k] = userdata.get(k)
os.environ["R2_BUCKET"] = "nepaldtm-g-1m"
BASE = "https://nepaldtm-data.achyutpdl.com/g/colab"
print("secrets loaded")

In [ ]:
!pip -q install rasterio shapely pyproj boto3 affine
!rm -rf /content/NepalDTM && git clone -q --depth 1 -b g https://github.com/achyut/NepalDTM /content/NepalDTM

Inputs: 8 m NepalDTM v2 for Bagmati on the national grid (2 GB, in parts), the province boundary and the model weights.

In [ ]:
%%bash
set -e
mkdir -p /content/in && cd /content/in
B=https://nepaldtm-data.achyutpdl.com/g/colab
curl -sSfL -A curl/8 -O $B/parts.txt -O $B/SHA256SUMS -O $B/bagmati.geojson -O $B/ema_150000.pt
for p in $(cat parts.txt); do curl -sSfL -A curl/8 -O $B/$p; done
cat $(cat parts.txt) > bagmati_v2_dtm_8m.tif && rm $(cat parts.txt)
sha256sum -c SHA256SUMS

## 1. Benchmark
Seconds per 256 m window at 10 steps × 2 samples for a few batch sizes. The laptop does 0.53 s per window.

In [ ]:
import sys, time, numpy as np, torch, rasterio
sys.path.insert(0, "/content/NepalDTM/code/g")
from g_model import DEV, UNet, sample
net = UNet().to(DEV); net.load_state_dict(torch.load("/content/in/ema_150000.pt", map_location=DEV)["ema"]); net.eval()
z = rasterio.open("/content/in/bagmati_v2_dtm_8m.tif").read(1, window=((8000, 8320), (16000, 16320))).astype("float32")
wins = torch.from_numpy(np.stack([z[i:i + 32, j:j + 32] for i in range(0, 288, 24) for j in range(0, 288, 24)]))[:, None]
best = None
for bs in (8, 32, 64, 128):
    x = wins[:bs].to(DEV).repeat_interleave(2, 0)
    try:
        with torch.no_grad():
            sample(net, x, n_steps=10); torch.cuda.synchronize(); t = time.time()
            for _ in range(3): sample(net, x, n_steps=10)
            torch.cuda.synchronize()
    except torch.OutOfMemoryError:  # smaller GPUs (A100 40 GB, L4): stop at the largest batch that fits
        print(f"batch {bs:4d}: out of GPU memory"); del x; torch.cuda.empty_cache(); break
    spw = (time.time() - t) / 3 / bs
    best = min(best or (9, 0), (spw, bs))
    print(f"batch {bs:4d}: {spw:.3f} s/window  ({0.53 / spw:.1f}x the laptop)")
BS = best[1]; print("use --bs", BS)
# free the GPU for the generation process (a separate process)
for v in ("net", "x", "wins"):
    globals().pop(v, None)
import gc; gc.collect(); torch.cuda.empty_cache()

## 2. Generate
Re-run this cell after any disconnect (after re-running the cells above); finished tiles are skipped.

In [ ]:
BS = globals().get('BS', 32)  # from the benchmark, or 32 if it was skipped (fits a 40 GB A100)
!cd /content/NepalDTM/code && python -u g/g_region.py --region Bagmati --reverse \
    --dtm /content/in/bagmati_v2_dtm_8m.tif --prov /content/in/bagmati.geojson --ckpt /content/in/ema_150000.pt \
    --steps 10 --ens 2 --bs {BS} --out /content/out --r2-prefix bagmati/tiles

Local copies in `/content/out/tiles` can be deleted once uploaded: `!rm /content/out/tiles/*.tif`.